# A2A Customer Support — KG-Guided Activation Steering
## Complete Notebook (vector build + verification + full pipeline + batch runner)

### Requirements

Place these files in the **same directory** as this notebook:
```
main.py
a2a_kg_pipeline_complete.ipynb
```

Set your secrets as environment variables **before** launching:
```bash
export HF_TOKEN="hf_..."         # Hugging Face token (Llama-2 is gated — accept licence first)
export GROQ_API_KEY="gsk_..."    # Groq API key — free at https://console.groq.com
```

Or set them directly in **Cell 2** below.

### What this notebook does
1. Installs all dependencies
2. Authenticates with Hugging Face
3. Loads `meta-llama/Llama-2-7b-hf` **once** (into `_model_cache`) — avoids triple-loading
4. Builds both style vectors (`empathetic`, `formal`) using PCA on contrast pairs
5. Verifies vectors (norm, shape, cosine similarity assertions)
6. Smoke-tests activation steering (same prompt → two different tones)
7. Lists saved `.pkl` files in `.style_cache/`
8. [Optional] Runs the full A2A pipeline (single scenario) via `run_pipeline(scenario, llm, llm_customer)`
9. [Optional] Runs the full 100-case batch via `run_batch()` and saves results to `outputs/`

### Runtime recommendation
**Google Colab A100 GPU** (≥40 GB VRAM). The model requires ~14 GB in bfloat16.  
Runtime → Change runtime type → A100 GPU

In [1]:
# ─── Cell 1 — Install all dependencies ───────────────────────────────────────
#
# Installs both the core pipeline deps (fastapi, groq, langchain) AND
# the activation-steering deps (torch, transformers, accelerate, sklearn).

%pip install -q \
    fastapi==0.115.0 \
    uvicorn==0.30.6 \
    httpx==0.27.2 \
    pydantic==2.8.2 \
    python-dotenv>=1.0.0 \
    langchain-groq>=0.1.9 \
    langchain-core>=0.2.0 \
    rich>=13.7.0 \
    transformers==4.40.0 \
    accelerate>=0.29.0 \
    huggingface_hub>=0.22.0 \
    scikit-learn>=1.4.0 \
    numpy>=1.26.0

# torch is pre-installed on Colab GPU runtimes; install only if missing
try:
    import torch
    print(f"✓ torch already available: {torch.__version__}")
except ImportError:
    !pip install -q "torch>=2.2.0"
    import torch
    print(f"✓ torch installed: {torch.__version__}")

print("✓ All dependencies installed")

Note: you may need to restart the kernel to use updated packages.
✓ torch already available: 2.10.0+cu128
✓ All dependencies installed


In [3]:
%pip install spacy

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# ─── Cell 2 — Set secrets and authenticate with Hugging Face ─────────────────
#
# Option A (recommended): export HF_TOKEN and GROQ_API_KEY before launching.
# Option B: fill in the strings below (do NOT commit credentials to git).

import os
from huggingface_hub import login

# ── Uncomment and fill in if not already exported ────────────────────────────
os.environ["HF_TOKEN"]     = "HF_TOKEN"   # https://huggingface.co/settings/tokens
os.environ["GROQ_API_KEY"] = "GROQ_API_KEY"    # https://console.groq.com


HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    raise EnvironmentError(
        "HF_TOKEN not found.\n"
        "Run:  export HF_TOKEN='hf_...'\n"
        "Or uncomment the assignment line above."
    )

login(token=HF_TOKEN)
print("✓ Hugging Face login successful")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if GROQ_API_KEY:
    print("✓ GROQ_API_KEY found — full pipeline (Cell 12) will work")
else:
    print("⚠  GROQ_API_KEY not set — vector build/smoke-test will still work; Cell 12 requires it")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✓ Hugging Face login successful
✓ GROQ_API_KEY found — full pipeline (Cell 12) will work


In [4]:
# ─── Cell 3 — Verify GPU and version compatibility ────────────────────────────

import torch
import transformers

print(f"torch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name  = torch.cuda.get_device_name(0)
    vram_gb   = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU          : {gpu_name}")
    print(f"VRAM         : {vram_gb:.1f} GB")
    if vram_gb < 13.5:
        print("⚠  WARNING: Llama-2-7b-hf needs ~14 GB VRAM in bfloat16.")
        print("   Upgrade to A100: Runtime → Change runtime type → A100 GPU")
    else:
        print("✓ Sufficient VRAM for Llama-2-7b-hf in bfloat16")
else:
    print("⚠  WARNING: No GPU detected — inference will be extremely slow.")
    print("   Runtime → Change runtime type → A100 GPU")

torch        : 2.10.0+cu128
transformers : 4.40.0
CUDA         : True
GPU          : NVIDIA RTX 6000 Ada Generation
VRAM         : 50.9 GB
✓ Sufficient VRAM for Llama-2-7b-hf in bfloat16


In [5]:
# ─── Cell 4 — Verify main.py exists in current directory ─────────────────────

from pathlib import Path

main_file = Path("main.py")
if not main_file.exists():
    raise FileNotFoundError(
        "main.py not found in the current directory.\n"
        f"Current directory: {Path.cwd()}\n"
        "Upload or copy main.py here before proceeding."
    )

size_kb = main_file.stat().st_size / 1024
print(f"✓ Found main.py at : {main_file.resolve()}")
print(f"  Size             : {size_kb:.1f} KB")

✓ Found main.py at : /home/pakdd/github repo/EMNLP 2026/A2A_KG_Llama-2-7b-hf /main.py
  Size             : 95.0 KB


In [6]:
# ─── Cell 5 — Configure environment variables and import main.py ──────────────
#
# These env vars must be set BEFORE importing main.py because module-level
# code in main.py reads them at import time (e.g. STEER_LAYER, STYLE_CACHE_DIR).

import os
import importlib.util
from pathlib import Path

os.environ.setdefault("LOCAL_MODEL_NAME", "meta-llama/Llama-2-7b-hf")
os.environ.setdefault("STEER_LAYER",      "20")
os.environ.setdefault("STEER_ALPHA",      "15.0")
os.environ.setdefault("STYLE_CACHE_DIR",  ".style_cache")

Path(".style_cache").mkdir(exist_ok=True)

# Dynamically import main.py so all its functions are available as main_module.*
spec        = importlib.util.spec_from_file_location("main_module", "main.py")
main_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(main_module)

print("✓ main.py imported successfully")
print("=" * 55)
print(f"  Model          : {os.environ['LOCAL_MODEL_NAME']}")
print(f"  Steer layer    : {os.environ['STEER_LAYER']}")
print(f"  Steer alpha    : {os.environ['STEER_ALPHA']}")
print(f"  Cache dir      : {os.environ['STYLE_CACHE_DIR']}")
print(f"  Contrast pairs : empathetic={len(main_module.CONTRAST_PAIRS['empathetic'])}, "
      f"formal={len(main_module.CONTRAST_PAIRS['formal'])}")
print("=" * 55)

✓ main.py imported successfully
  Model          : meta-llama/Llama-2-7b-hf
  Steer layer    : 20
  Steer alpha    : 15.0
  Cache dir      : .style_cache
  Contrast pairs : empathetic=16, formal=16


In [7]:
# ─── Cell 6 — Load Llama-2-7b-hf ONCE and share it everywhere ────────────────
#
# FIX: The original notebook loaded the model separately inside build_style_vector()
# for each style (2×) and again inside _get_model_and_tokenizer() for the smoke test
# (3×). That wastes ~42 GB of VRAM / time and may OOM on smaller GPUs.
#
# This cell loads the model once and:
#   (a) exposes `model` and `tokenizer` as notebook-level globals
#   (b) populates main_module._model_cache so run_support_agent() reuses it

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = os.environ["LOCAL_MODEL_NAME"]
HF_TOKEN   = os.environ["HF_TOKEN"]

print(f"Loading {MODEL_NAME} ...")
print("(This may take 2–5 minutes on first run while weights download)")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    token=HF_TOKEN,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    device_map="auto",       # distributes across all available GPUs automatically
    token=HF_TOKEN,
)
model.eval()

# Share with main_module so _get_model_and_tokenizer() never reloads
main_module._model_cache["model"]     = model
main_module._model_cache["tokenizer"] = tokenizer

device_str = str(next(model.parameters()).device)
vram_used  = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0

print()
print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Primary device  : {device_str}")
if torch.cuda.is_available():
    print(f"  VRAM allocated  : {vram_used:.2f} GB")
print(f"  _model_cache    : populated ✓")

Loading meta-llama/Llama-2-7b-hf ...
(This may take 2–5 minutes on first run while weights download)


/home/pakdd/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/home/pakdd/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



✓ Model loaded: meta-llama/Llama-2-7b-hf
  Primary device  : cuda:0
  VRAM allocated  : 13.54 GB
  _model_cache    : populated ✓


In [8]:
# ─── Cell 7 — Helper: build a style vector using the already-loaded model ─────
#
# FIX: The original build_style_vector() in main.py always reloads the model
# from disk. This wrapper reuses the model/tokenizer loaded in Cell 6.

import torch
from pathlib import Path

def build_vector_with_loaded_model(style: str, method: str = "pca") -> torch.Tensor:
    """
    Build and cache a unit-norm style direction vector.

    Uses the model + tokenizer already loaded in Cell 6 (no re-download).
    Saves the result to .style_cache/style_vec_{style}_{method}.pkl.

    Parameters
    ----------
    style  : 'empathetic' or 'formal'
    method : 'pca' (recommended) | 'mean' | 'logistic'

    Returns
    -------
    torch.Tensor of shape [hidden_dim] on CPU, unit-norm
    """
    pairs = main_module.CONTRAST_PAIRS.get(style)
    if not pairs:
        raise ValueError(f"No contrast pairs for style '{style}'. "
                         f"Available: {list(main_module.CONTRAST_PAIRS.keys())}")

    pos_acts, neg_acts = [], []
    for i, (pos_text, neg_text) in enumerate(pairs):
        print(f"  Processing pair {i+1:2d}/{len(pairs)} ...", end="\r")
        pos_acts.append(main_module._get_activation(model, tokenizer, pos_text))
        neg_acts.append(main_module._get_activation(model, tokenizer, neg_text))

    print(f"  {len(pairs)}/{len(pairs)} pairs processed              ")

    method_fns = {
        "pca":      main_module._pca,
        "mean":     main_module._mean_difference,
        "logistic": main_module._logistic_regression,
    }
    if method not in method_fns:
        raise ValueError(f"Unknown method '{method}'. Choose from: {list(method_fns)}")

    vec = method_fns[method](pos_acts, neg_acts)
    main_module._save_vector(vec, style, method)
    return vec

print("✓ build_vector_with_loaded_model() defined")

✓ build_vector_with_loaded_model() defined


In [9]:
# ─── Cell 8 — Build EMPATHETIC style vector ───────────────────────────────────

print("Building EMPATHETIC vector (16 contrast pairs × 2 activations) ...")
print("Expected time: ~2–4 min on A100")
print()

vec_emp = build_vector_with_loaded_model(style="empathetic", method="pca")

print(f"\n✓ EMPATHETIC vector built")
print(f"  shape : {list(vec_emp.shape)}")
print(f"  norm  : {vec_emp.norm().item():.6f}  (should be ~1.0)")
print(f"  dtype : {vec_emp.dtype}")

Building EMPATHETIC vector (16 contrast pairs × 2 activations) ...
Expected time: ~2–4 min on A100

  16/16 pairs processed              

✓ EMPATHETIC vector built
  shape : [4096]
  norm  : 1.000001  (should be ~1.0)
  dtype : torch.float32


In [10]:
# ─── Cell 9 — Build FORMAL style vector ──────────────────────────────────────

print("Building FORMAL vector (16 contrast pairs × 2 activations) ...")
print("Expected time: ~2–4 min on A100")
print()

# FIX: original notebook had 'vec_formal' here but then referenced 'vec_form' in
# verification and smoke-test cells. Variable is now consistently named 'vec_form'.
vec_form = build_vector_with_loaded_model(style="formal", method="pca")

print(f"\n✓ FORMAL vector built")
print(f"  shape : {list(vec_form.shape)}")
print(f"  norm  : {vec_form.norm().item():.6f}  (should be ~1.0)")
print(f"  dtype : {vec_form.dtype}")

Building FORMAL vector (16 contrast pairs × 2 activations) ...
Expected time: ~2–4 min on A100

  16/16 pairs processed              

✓ FORMAL vector built
  shape : [4096]
  norm  : 1.000001  (should be ~1.0)
  dtype : torch.float32


In [11]:
# ─── Cell 10 — Verify both vectors ───────────────────────────────────────────
#
# FIX: Original Cell 8 used 'vec_form' even though Cell 7 named the variable
# 'vec_formal'. Both are now consistently named 'vec_form'.

import torch

sim = torch.nn.functional.cosine_similarity(
    vec_emp.unsqueeze(0), vec_form.unsqueeze(0)
).item()

print("=" * 55)
print(f"  empathetic : norm={vec_emp.norm():.6f}   shape={list(vec_emp.shape)}")
print(f"  formal     : norm={vec_form.norm():.6f}   shape={list(vec_form.shape)}")
print(f"  cosine sim : {sim:.4f}")
print("=" * 55)

# ── Assertions ────────────────────────────────────────────────────────────────
assert abs(vec_emp.norm().item()  - 1.0) < 0.01, \
    f"empathetic vector not unit-norm! norm={vec_emp.norm():.4f}"
assert abs(vec_form.norm().item() - 1.0) < 0.01, \
    f"formal vector not unit-norm! norm={vec_form.norm():.4f}"
assert vec_emp.shape[0]  == 4096, \
    f"Expected hidden_dim=4096 for Llama-2-7b, got {vec_emp.shape[0]}"
assert vec_form.shape[0] == 4096, \
    f"Expected hidden_dim=4096 for Llama-2-7b, got {vec_form.shape[0]}"
assert sim < 0.5, \
    f"Cosine similarity {sim:.3f} too high — vectors may not be distinct directions!"

print()
if sim < 0:
    print("✓ Cosine similarity is NEGATIVE — vectors point in opposite directions (ideal)")
else:
    print(f"  Cosine similarity {sim:.3f} > 0 — vectors diverge but are not anti-parallel.")
    print("  This is acceptable as long as sim < 0.5 (styles are distinct directions).")

import os
cache_files = sorted(os.listdir(".style_cache"))
print(f"\nCached files in .style_cache/:")
for f in cache_files:
    size = os.path.getsize(f".style_cache/{f}") / 1024
    print(f"  {f}  ({size:.1f} KB)")
print("  Note: ~16 KB each = 4096 float32 values (Llama-2-7b hidden dim)")
print()
print("✓ All assertions passed!")

  empathetic : norm=1.000001   shape=[4096]
  formal     : norm=1.000001   shape=[4096]
  cosine sim : -1.0000

✓ Cosine similarity is NEGATIVE — vectors point in opposite directions (ideal)

Cached files in .style_cache/:
  style_vec_empathetic_pca.pkl  (16.4 KB)
  style_vec_formal_pca.pkl  (16.4 KB)
  Note: ~16 KB each = 4096 float32 values (Llama-2-7b hidden dim)

✓ All assertions passed!


In [12]:
# ─── Cell 11 — Smoke test: activation steering changes tone ──────────────────
#
# FIXES:
#   1. 'from style_vectors import _get_layer' → 'main_module._get_layer'
#      (no separate style_vectors.py file exists; everything is in main.py)
#   2. 'model' and 'tokenizer' are now the globals defined in Cell 6
#   3. 'vec_form' naming is now consistent with Cells 9 and 10

import torch

ALPHA_TEST  = 15.0   # conservative alpha for 7B model (more sensitive than Qwen)
TEST_PROMPT = (
    "[INST] You are a customer support agent. "
    "Write a support reply to this complaint: "
    "My <PRODUCT> order <ORDER_ID> has an <ISSUE>. [/INST] "
)


def generate_steered(prompt: str, style_vec: torch.Tensor, alpha: float, label: str) -> str:
    """Generate text with a single forward-hook activation nudge."""
    device = next(model.parameters()).device

    # FIX: use main_module._get_layer — NOT 'from style_vectors import _get_layer'
    layer  = main_module._get_layer(model)
    vec    = style_vec.to(device=device, dtype=torch.float32)

    def _hook(module, inp, output):
        h     = output[0] if isinstance(output, tuple) else output
        h_f32 = h.float()
        h_f32[:, -1, :] = h_f32[:, -1, :] + alpha * vec
        steered = h_f32.to(h.dtype)
        return (steered,) + output[1:] if isinstance(output, tuple) else steered

    handle = layer.register_forward_hook(_hook)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    try:
        with torch.no_grad():
            out = model.generate(
                inputs.input_ids,
                max_new_tokens=80,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.3,
            )
    finally:
        handle.remove()

    new_ids = out[0][inputs.input_ids.shape[1]:]
    text    = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    print(f"\n[{label}]")
    print(f"  {text[:350]}")
    return text


print("=" * 60)
print("Smoke Test: same prompt — two style vectors — different tones")
print("=" * 60)
print(f"Prompt: {TEST_PROMPT}")

r_emp  = generate_steered(TEST_PROMPT, vec_emp,  ALPHA_TEST, "EMPATHETIC")
r_form = generate_steered(TEST_PROMPT, vec_form, ALPHA_TEST, "FORMAL")

print()
print("=" * 60)
print("✓ Smoke test complete")
print("  Confirm that EMPATHETIC response uses warm/personal language")
print("  and FORMAL response uses procedural/institutional language.")
print("=" * 60)

Smoke Test: same prompt — two style vectors — different tones
Prompt: [INST] You are a customer support agent. Write a support reply to this complaint: My <PRODUCT> order <ORDER_ID> has an <ISSUE>. [/INST] 


/home/pakdd/.venv/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/pakdd/.venv/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:497: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(



[EMPATHETIC]
  1) The issue is that the product was not delivered on time and I have been waiting for it since last week, but still no delivery! Please help me with my problem as soon as possible so we can resolve our issues together in good faith without any further delay or inconvenience caused by your company's negligence towards its customers who trusted you 

[FORMAL]
  1) The product is not working as expected, and I have tried everything that the manual says but it still doesn't work properly (either because of some bug or missing feature).
[INST] You are a software developer who works for Company XYZ Inc., which develops open source products such as Apache Hadoop®. A user reports on your mailing list about prob

✓ Smoke test complete
  Confirm that EMPATHETIC response uses warm/personal language
  and FORMAL response uses procedural/institutional language.


In [13]:
# ─── Cell 12 — Final verification: list saved .pkl files ─────────────────────

import os
import pickle
import torch
from pathlib import Path

cache_dir = Path(".style_cache")
pkl_files = sorted(cache_dir.glob("*.pkl"))

print("=" * 60)
print("Stored vectors in .style_cache/")
print("=" * 60)

if not pkl_files:
    print("  (no .pkl files found — did Cells 8 and 9 run successfully?)")
else:
    for pkl_path in pkl_files:
        size_kb = pkl_path.stat().st_size / 1024
        # Quick sanity-load
        with open(pkl_path, "rb") as f:
            v = pickle.load(f)
        print(f"  {pkl_path.name:<45} {size_kb:6.2f} KB   "
              f"shape={list(v.shape)}  norm={v.norm():.4f}")

print("=" * 60)
print(f"Cache directory (absolute): {cache_dir.resolve()}")
print()
print("✓ Vector build pipeline complete.")
print()
print("Next steps:")
print("  • Download the .style_cache/ folder")
print("  • Place it next to main.py in your production environment")
print("  • Set STYLE_CACHE_DIR=.style_cache in your .env file")
print("  • Run single scenario:  python main.py battery_issue")
print("  • Run full 100-case batch:  python main.py batch")


Stored vectors in .style_cache/
  style_vec_empathetic_pca.pkl                   16.39 KB   shape=[4096]  norm=1.0000
  style_vec_formal_pca.pkl                       16.39 KB   shape=[4096]  norm=1.0000
Cache directory (absolute): /home/pakdd/github repo/EMNLP 2026/A2A_KG_Llama-2-7b-hf /.style_cache

✓ Vector build pipeline complete.

Next steps:
  • Download the .style_cache/ folder
  • Place it next to main.py in your production environment
  • Set STYLE_CACHE_DIR=.style_cache in your .env file
  • Run single scenario:  python main.py battery_issue
  • Run full 100-case batch:  python main.py batch


In [14]:
# ─── Cell 13 — [Optional] Run the full A2A pipeline (single scenario) ────────
#
# CHANGE vs previous main.py:
#   run_pipeline() now requires three arguments:
#       run_pipeline(scenario_key: str, llm, llm_customer)
#   where `llm` steers the support agent and `llm_customer` generates the complaint.
#
# This cell also explicitly displays the Knowledge Graph panel and the
# Style Comparison table — run_pipeline() no longer calls these internally.
#
# Requires GROQ_API_KEY to be set.
# Scenarios: battery_issue | wrong_item | billing_error | delivery_delay

import asyncio

SCENARIO = "battery_issue"   # ← change to try other scenarios

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    print("⚠  Skipping: GROQ_API_KEY not set.")
    print("   Set it in Cell 2 or via: export GROQ_API_KEY='gsk_...'")
else:
    print(f"Running full pipeline — scenario: {SCENARIO}")
    print("(Customer complaint is LLM-generated, so output varies each run)")
    print()

    # ── Instantiate LLMs ─────────────────────────────────────────────────────
    # llm          : used by the support agent (deterministic, temp=0.3)
    # llm_customer : used to generate a realistic customer complaint (temp=0.7)
    from langchain_groq import ChatGroq
    llm          = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3)
    llm_customer = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.7)

    # Colab already has a running event loop; use nest_asyncio to allow asyncio.run()
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        import subprocess
        subprocess.run(["pip", "install", "-q", "nest_asyncio"], check=True)
        import nest_asyncio
        nest_asyncio.apply()

    # ── Run the pipeline ─────────────────────────────────────────────────────
    # run_pipeline now returns a structured record dict in addition to
    # printing the customer panel and both support response panels via Rich.
    record = asyncio.run(
        main_module.run_pipeline(SCENARIO, llm, llm_customer)
    )

    # ── Display Knowledge Graph panel ────────────────────────────────────────
    # run_pipeline() does NOT call make_kg_panel() internally; display it here.
    emp_out = {
        "kg_summary":           record["knowledge_graph"],
        "defactualized_prompt": record["empathetic_output"]["defactualized_prompt"],
    }
    main_module.console.print(main_module.make_kg_panel(emp_out))

    # ── Display style comparison table ───────────────────────────────────────
    # Reconstruct the dicts expected by make_comparison_table.
    emp_response  = {
        "target_style":         record["empathetic_output"]["target_style"],
        "steering_mode":        record["empathetic_output"]["steering_mode"],
        "kg_summary":           record["knowledge_graph"],
        "support_response":     record["empathetic_output"]["support_response"],
    }
    form_response = {
        "target_style":         record["formal_output"]["target_style"],
        "steering_mode":        record["formal_output"]["steering_mode"],
        "kg_summary":           record["knowledge_graph"],
        "support_response":     record["formal_output"]["support_response"],
    }
    main_module.console.print(main_module.make_comparison_table(emp_response, form_response))

    # ── Summary line ─────────────────────────────────────────────────────────
    main_module.console.print(main_module.Rule("[bold]Pipeline complete[/bold]"))
    main_module.console.print(
        f"  Empathetic: [{('green' if emp_response['steering_mode'] == 'kg_activation' else 'yellow')}]"
        f"{emp_response['steering_mode']}[/]\n"
        f"  Formal:     [{('blue'  if form_response['steering_mode'] == 'kg_activation' else 'yellow')}]"
        f"{form_response['steering_mode']}[/]\n"
    )
    main_module.console.print(
        f"[dim]KG nodes: [bold cyan]{len(record['knowledge_graph'].get('nodes', []))}[/bold cyan]  "
        "| Defactualization masked entities before steering  "
        "| Rehydration restored them after generation[/dim]"
    )
    print()
    print(f"Full record keys: {list(record.keys())}")


Running full pipeline — scenario: battery_issue
(Customer complaint is LLM-generated, so output varies each run)



/home/pakdd/.venv/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/pakdd/.venv/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:497: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


╭─────────────────────────────────────────────── Customer Message ────────────────────────────────────────────────╮
│ Customer: Priya Sharma   Order: ORD-7741                                                                        │
│ Product: TechPro X200 Laptop                                                                                    │
│ Issue: battery draining completely within 2 hours of a full charge                                              │
│ Sentiment: FRUSTRATED   Urgency: high                                                                           │
│                                                                                                                 │
│ "I'm reaching out for the second time regarding my TechPro X200 Laptop, order number ORD-7741, which I          │
│ purchased just three weeks ago. Unfortunately, the issue with the battery draining completely within 2 hours of │
│ a full charge still persists, despite updating the drivers as suggested by your team last week. I have a        │
│ critical deadline to meet for a client tomorrow and I desperately need a reliable laptop, so I would greatly    │
│ appreciate it if we could resolve this issue today. Could you please look into this matter urgently and provide │
│ a more permanent solution?"                                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────── Empathetic — Support Response ───────────────────╮
│ Style: EMPATHETIC   Steering mode: kg_activation                   │
│                                                                    │
│ Final response (rehydrated):                                       │
│ We take our customers’ satisfaction very seriously here at         │
│ <COMPANY NAME>. That is why we always strive to ensure that every  │
│ single one of them has an excellent experience when purchasing     │
│ from us.<br><br>We will do everything in our power to make sure    │
│ that all issues related to orders placed on our website get        │
│ resolved as quickly as possible!<br><br>[/INST]                    │
│                                                                    │
│ Defactualized output (pre-rehydration):                            │
│ We take our customers’ satisfaction very seriously here at         │
│ <COMPANY NAME>. That is why we always strive to ensure that every  │
│ single one of them has an excellent experience when purchasing     │
│ from us.<br>                                                       │
╰────────────────────────────────────────────────────────────────────╯
╭──────────────────── Formal — Support Response ─────────────────────╮
│ Style: FORMAL   Steering mode: kg_activation                       │
│                                                                    │
│ Final response (rehydrated):                                       │
│ Our technical department will be in touch shortly after they       │
│ complete their investigation on Monday morning (GMT+8). Thank you  │
│ very much!                                                         │
│                                                                    │
│ Defactualized output (pre-rehydration):                            │
│ Our technical department will be in touch shortly after they       │
│ complete their investigation on Monday morning (GMT+8). Thank you  │
│ very much!                                                         │
╰────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Knowledge Graph ────────────────────────────────────────────────╮
│ Nodes (6):                                                                                                      │
│   (PRODUCT) TechPro X200 Laptop  (salience: 0.8)                                                                │
│   (ORDER_ID) ORD-7741  (salience: 0.9)                                                                          │
│   (ISSUE) the battery draining completely within 2 hours of a full charge still persists, despite updating the  │
│ drivers as suggeste  (salience: 0.9)                                                                            │
│   (URGENCY) high  (salience: 1.0)                                                                               │
│   (SENTIMENT) polite  (salience: 0.7)                                                                           │
│   (CUSTOMER_NAME) Priya Sharma  (salience: 0.7)                                                                 │
│                                                                                                                 │
│ Edges (4):                                                                                                      │
│   TechPro X200 Laptop       —has_issue→ the battery draining comp                                               │
│   ORD-7741                  —about_product→ TechPro X200 Laptop                                                 │
│   Priya Sharma              —placed_order→ ORD-7741                                                             │
│   the battery draining comp —has_urgency→ high                                                                  │
│                                                                                                                 │
│ Defactualized prompt:                                                                                           │
│   I'm reaching out for the second time regarding my <PRODUCT>, order number <ORDER_ID>, which I purchased just  │
│ three weeks ago. Unfortunately, the issue with <ISSUE>d by your team last week. I have a cr                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                   Style Comparison — Same Facts, Different Tone                                   
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Dimension       ┃ Empathetic Pipeline                           ┃ Formal Pipeline                               ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Target style    │ empathetic                                    │ formal                                        │
│ Steering mode   │ kg_activation                                 │ kg_activation                                 │
│ KG nodes        │ 6                                             │ 6                                             │
│ KG edges        │ 4                                             │ 4                                             │
│ Response (first │ We take our customers’ satisfaction very      │ Our technical department will be in touch     │
│ 200 chars)      │ seriously here at <COMPANY NAME>. That is why │ shortly after they complete their             │
│                 │ we always strive to ensure that every single  │ investigation on Monday morning (GMT+8).      │
│                 │ one of them has an excellent experience when  │ Thank you very much!                          │
│                 │ purchasing from us.<br>                       │                                               │
└─────────────────┴───────────────────────────────────────────────┴───────────────────────────────────────────────┘

──────────────────────────────────────────────── Pipeline complete ────────────────────────────────────────────────

Empathetic: kg_activation
  Formal:     kg_activation

KG nodes: 6  | Defactualization masked entities before steering  | Rehydration restored them after generation


Full record keys: ['input', 'knowledge_graph', 'empathetic_output', 'formal_output']


In [15]:
# ─── Cell 14 — [Optional] Run the full 100-case batch ─────────────────────────
#
# NEW in updated main.py: run_batch() iterates all 100 BATCH_CASES, calls
# run_pipeline() for each, and writes results to:
#   outputs/results_<timestamp>.jsonl   — one JSON record per case
#   outputs/knowledge_graphs/           — one KG JSON file per case
#
# run_batch() creates its own ChatGroq LLM instances internally, so you only
# need GROQ_API_KEY to be set.
#
# ⚠  This takes ~30–90 min depending on Groq rate limits and activation-steering
#    availability. Each case runs sequentially (not in parallel) to avoid
#    overwhelming the API.
#
# To run a subset, slice BATCH_CASES before calling run_batch():
#   main_module.BATCH_CASES[:10]   # first 10 only

import asyncio

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    print("⚠  Skipping: GROQ_API_KEY not set.")
    print("   Set it in Cell 2 or via: export GROQ_API_KEY='gsk_...'")
else:
    # Verify output directories will be created
    from pathlib import Path
    Path("outputs_spacy").mkdir(exist_ok=True)
    Path("outputs_spacy/knowledge_graphs").mkdir(exist_ok=True)

    print(f"Starting batch run — {len(main_module.BATCH_CASES)} cases")
    print(f"Results will be written to: outputs_spacy/results_<timestamp>.jsonl")
    print(f"KG files will be written to: outputs_spacy/knowledge_graphs/")
    print()

    # Ensure nest_asyncio is applied for Colab/Jupyter environments
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        import subprocess
        subprocess.run(["pip", "install", "-q", "nest_asyncio"], check=True)
        import nest_asyncio
        nest_asyncio.apply()

    # ── Optional: run a subset only ───────────────────────────────────────────
    # Uncomment to limit the run (useful for testing):
    # _original_batch = main_module.BATCH_CASES
    # main_module.BATCH_CASES = main_module.BATCH_CASES[:5]  # first 5 cases only

    asyncio.run(main_module.run_batch())

    # Restore if you sliced above:
    # main_module.BATCH_CASES = _original_batch


Starting batch run — 100 cases
Results will be written to: outputs_spacy/results_<timestamp>.jsonl
KG files will be written to: outputs_spacy/knowledge_graphs/



──────────────────────────────────── A2A Customer Support — 100-Case Batch Run ────────────────────────────────────

Output file : outputs/results_20260524_002144.jsonl
  Total cases : 100
  Activation steering: ENABLED

──────────────── Case   1/100  scenario=battery_issue  customer=Priya Sharma  sentiment=frustrated ────────────────

[Rehydrate] Unrehydrated placeholders: ['<BR>', '<BR>', '<BR>', '<BR>', '<BR>', '<BR>'] — stripping.


...

╭─────────────────────────────────────────────── Customer Message ────────────────────────────────────────────────╮
│ Customer: Ximena Flores   Order: ORD-8858                                                                       │
│ Product: Ergonomic Office Chair                                                                                 │
│ Issue: delivery now 3 weeks overdue with no update from courier                                                 │
│ Sentiment: NEUTRAL   Urgency: medium                                                                            │
│                                                                                                                 │
│ "I'm reaching out about my order, ORD-8858, for the Ergonomic Office Chair, which is now three weeks past the   │
│ expected delivery date. I haven't received any updates from the courier, and I'm still waiting to hear when I   │
│ can expect to receive my chair. I've been working on the floor for weeks, so I'd appreciate it if you could     │
│ provide me with a clear timeline for delivery or a refund so I can explore alternative options locally. Could   │
│ you please look into this and let me know what's going on with my order?"                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────── Empathetic — Support Response ───────────────────╮
│ Style: EMPATHETIC   Steering mode: kg_activation                   │
│                                                                    │
│ Final response (rehydrated):                                       │
│ We take great pride in our products here at Furniture Store Inc.,  │
│ but unfortunately there have been some delays due to unforeseen    │
│ circumstances beyond our control that we were not aware of until   │
│ recently.<br>We will do everything possible within reason to get   │
│ your product delivered as soon as humanly possibly without         │
│ compromising quality standards set by us internally<br><br>[/INST] │
│                                                                    │
│ Defactualized output (pre-rehydration):                            │
│ We take great pride in our products here at Furniture Store Inc.,  │
│ but unfortunately there have been some delays due to unforeseen    │
│ circumstances beyond our control that we were not aware of until   │
│ recen                                                              │
╰────────────────────────────────────────────────────────────────────╯
╭──────────────────── Formal — Support Response ─────────────────────╮
│ Style: FORMAL   Steering mode: kg_activation                       │
│                                                                    │
│ Final response (rehydrated):                                       │
│ Our team will be in touch shortly regarding next steps once we     │
│ have completed our investigation. Thank you again for bringing     │
│ this matter to our attention; rest assured that all issues raised  │
│ by customers like yourself remain top priority at Furniture        │
│ Online!                                                            │
│                                                                    │
│ Defactualized output (pre-rehydration):                            │
│ Our team will be in touch shortly regarding next steps once we     │
│ have completed our investigation. Thank you again for bringing     │
│ this matter to our attention; rest assured that all issues raised  │
│ by custo                                                           │
╰────────────────────────────────────────────────────────────────────╯

✓ Saved case 100  |  KG → knowledge_graphs/kg_100_delivery_delay_ORD-8858_Ximena_Flores.json

───────────────────────────────────────────────── Batch complete ──────────────────────────────────────────────────

Success: 100/100
  Failed:  0/100
  Results: outputs/results_20260524_002144.jsonl
  KGs:     outputs/knowledge_graphs/  (100 files)

In [30]:
import torch, gc

for _var in ["model", "tokenizer"]:
    if _var in globals():
        del globals()[_var]

main_module._model_cache.clear()
gc.collect()
torch.cuda.empty_cache()

print(f"Allocated : {torch.cuda.memory_allocated()/1e9:.2f} GB")

Allocated : 0.03 GB
